# 01 · Lyden

## Opsætning

In [ ]:
import sys
from pathlib import Path

# VS Code runs a notebook in its own folder: walk up to training/, the folder with pyproject.toml.
TRAINING = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "pyproject.toml").is_file() and (p / "wt_train").is_dir())
sys.path.insert(0, str(TRAINING))

import warnings

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio

warnings.filterwarnings("ignore", message="IProgress not found")  # a harmless notice: no progress widgets
from transformers import WhisperFeatureExtractor

from wt_train import audio, fleurs, paths

## Et klip fra FLEURS

In [ ]:
CLIP_ID = "fleurs-da_dk-1480530483950034476"  # the same clip as in 02
clip = next(c for c in fleurs.load_split("test") if c.id == CLIP_ID)
pcm = audio.read_16k_mono(clip.wav)  # float32 samples, 16 kHz mono

print("Tekst:   ", clip.text)
print("Samples: ", len(pcm), "=", len(pcm) / audio.SAMPLE_RATE, "sekunder")
Audio(pcm, rate=audio.SAMPLE_RATE)

## Lydbølgen

In [ ]:
seconds = np.arange(len(pcm)) / audio.SAMPLE_RATE

fig, ax = plt.subplots(figsize=(12, 3), layout="constrained")
ax.plot(seconds, pcm, linewidth=0.4)
ax.set(title=f"Lydbølgen: {audio.SAMPLE_RATE} tal i sekundet", xlabel="Tid (sekunder)",
       ylabel="Udsving", xlim=(0, seconds[-1]))
plt.show()

## Eddas log-mel-spektrogram

In [ ]:
fe = WhisperFeatureExtractor.from_pretrained(paths.snapshot_dir(paths.EDDA_REPO, paths.EDDA_REVISION))
features = fe(pcm, sampling_rate=fe.sampling_rate, return_tensors="np").input_features

print("Form:", features.shape, "= klip × mel-bånd × tidsskridt")
print("Et tidsskridt:", 1000 * fe.hop_length / fe.sampling_rate, "ms;  vinduet:", fe.chunk_length,
      "sekunder;  udfyldning:", fe.n_samples - len(pcm), "nuller")

end = len(pcm) / fe.sampling_rate  # where the recording ends and the padding starts
fig, axes = plt.subplots(2, 1, figsize=(12, 7), layout="constrained")
for ax, right in zip(axes, [fe.chunk_length, end]):
    im = ax.imshow(features[0], origin="lower", aspect="auto", cmap="magma", interpolation="nearest",
                   extent=(0, fe.chunk_length, 0, fe.feature_size))
    ax.axvline(end, color="white", linestyle="--")
    ax.set(xlim=(0, right), xlabel="Tid (sekunder)", ylabel="Mel-bånd (lav til høj tone)")
axes[0].text(end + 0.3, fe.feature_size - 12, f"optagelsen slutter ({end:.1f} s), resten er udfyldning".replace(".", ","),
             color="white")
axes[0].set_title(f"Det Edda hører: {features.shape[1]} × {features.shape[2]} tal, altid {fe.chunk_length} sekunder")
axes[1].set_title("Kun optagelsen")
fig.colorbar(im, ax=axes, label="Styrke (log)")
plt.show()

## Samme spektrogram i hånden

In [ ]:
x = np.pad(pcm, (0, fe.n_samples - len(pcm)))  # zeros up to 30 s
x = np.pad(x, fe.n_fft // 2, mode="reflect")  # centre each window on its time step
frames = np.lib.stride_tricks.sliding_window_view(x, fe.n_fft)[::fe.hop_length]  # 25 ms windows every 10 ms
window = np.hanning(fe.n_fft + 1)[:-1]  # periodic Hann window
power = np.abs(np.fft.rfft(frames * window)) ** 2  # frames × 201 frequencies
mel = power[:-1] @ fe.mel_filters  # 3000 × 128 mel bands (the last frame is dropped)
logmel = np.log10(np.maximum(mel, 1e-10))
logmel = np.maximum(logmel, logmel.max() - 8.0)  # nothing quieter than 80 dB below the loudest
logmel = (logmel + 4.0) / 4.0

print("Form:", logmel.T.shape, " største forskel fra Eddas egen:", np.abs(logmel.T - features[0]).max())